In [1]:
!pip install --upgrade -q google-genai


In [2]:
import os
from google import genai

In [3]:
# Set up with API key
from google.colab import userdata
API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=API_KEY)

In [4]:
MODEL_ID = "gemini-2.5-flash"

In [5]:
# Generate a response with function calling
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Who is the current president of the US",
)
# Access the generated text and print it
print(response.text)

The current president of the United States is **Joe Biden**.


In [6]:
!pip install -U \
  "langchain==0.3.7" \
  "langchain-core==0.3.15" \
  "langchain-community" \
  "langgraph==0.2.35" \
  "langchain-google-genai" \
  "google-search-results"

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_google_genai-4.2.2-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.30-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.29-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.28-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.26-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take 

In [7]:
# Imports
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SerpAPIWrapper

import os
from google.colab import userdata


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [8]:
# Set environment variables
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get('SERPAPI_API_KEY')


In [9]:
# Initialize Gemini 1.5 Pro (latest compatible with LangChain)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [10]:
# SerpAPI search utility
search = SerpAPIWrapper()  # reads SERPAPI_API_KEY from env

# Wrap it as a LangChain Tool
tools = [
    Tool(
        name="search",
        func=search.run,
        description=(
            "Use this tool to search the web for up-to-date information, "
            "news, and current events."
        ),
    )
]


In [11]:
from langchain_core.prompts import PromptTemplate

react_template = """
You are a real-time search assistant.
You have access to the following tools:
{tools}

Tool names: {tool_names}

Use the search tool at most once.
Verify that results are current (year 2026).
If search data seems outdated, say: "Search results outdated".
Then answer using your best, up-to-date knowledge.

Follow this format:

Question: the user question
Thought: you should always think about what to do
Action: the tool to use, one of [{tool_names}]
Action Input: the input to the tool
Observation: the result of the tool
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the answer to the user

Begin!

Question: {input}
{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(react_template)


# Create ReAct agent with Gemini + search tool
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

# Agent runtime
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)


In [12]:
result = agent_executor.invoke(
    {"input": "Who is the president of the US"}
)

print(result["output"])




> Entering new AgentExecutor chain...
Action: search
Action Input: current president of US['Donald Trump is the 47th and current president since January 20, 2025.', 'President Donald J. Trump is returning to the White House to build upon his previous successes and use his mandate to reject the extremist policies.', 'President Donald J. Trump. 11091175 likes · 19820 talking about this. 45th & 47th President of the United States. The Golden Age of America Begins...', 'Learn about the duties of the U.S. president, vice president ... The 47th and current president of the United States is Donald John Trump.', 'Donald J. Trump is the 45th President of the United States. He believes the United States has incredible potential and will go on to exceed even its remarkable ...', "Trump is President of the United States and is a Republican. He has served since Jan. 20, 2025. Trump's current term ends on Jan. 20, 2029.", 'Barack Obama2009 to 2017 · George W. Bush photo. George W. Bush2001 to 2009